# Build 0-9 Digit Gesture Dataset
Split the downloaded Sign Language Digits dataset into train, test, and validation folders, then convert images to 64x64 grayscale for the CNN input.


In [ ]:
# Source dataset -------------------------------------------------------------
'''
Sign-Language-Digits-Dataset/
    Dataset/
        0/
            IMG_*.JPG
        1/
            IMG_*.JPG
        ...
        9/
            IMG_*.JPG

Generated dataset:

dataset_digits_gray64/
    train/
        0/ ... 9/
    validation/
        0/ ... 9/
    test/
        0/ ... 9/
'''


In [1]:
from pathlib import Path
import random
import shutil

import cv2

project_root = Path('.') if Path('Sign-Language-Digits-Dataset/Dataset').exists() else Path('..')
source_root = project_root / 'Sign-Language-Digits-Dataset/Dataset'
target_root = project_root / 'dataset_digits_gray64'
classes = [str(i) for i in range(10)]
split_ratio = {'train': 0.70, 'validation': 0.15, 'test': 0.15}
img_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
image_size = 64
seed = 6350
clean_target = True

if not source_root.exists():
    raise FileNotFoundError(f'Source dataset not found: {source_root.resolve()}')

missing_classes = [c for c in classes if not (source_root / c).is_dir()]
if missing_classes:
    raise FileNotFoundError(f'Missing class folders: {missing_classes}')

if clean_target and target_root.exists():
    shutil.rmtree(target_root)

for split in split_ratio:
    for c in classes:
        (target_root / split / c).mkdir(parents=True, exist_ok=True)

rng = random.Random(seed)
counts = {split: {c: 0 for c in classes} for split in split_ratio}
failures = []

for c in classes:
    files = [p for p in sorted((source_root / c).iterdir()) if p.suffix.lower() in img_ext]
    rng.shuffle(files)

    n_total = len(files)
    n_train = round(n_total * split_ratio['train'])
    n_validation = round(n_total * split_ratio['validation'])

    split_files = {
        'train': files[:n_train],
        'validation': files[n_train:n_train + n_validation],
        'test': files[n_train + n_validation:],
    }

    for split, paths in split_files.items():
        for src in paths:
            img = cv2.imread(str(src), cv2.IMREAD_GRAYSCALE)
            if img is None:
                failures.append(src)
                continue

            img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_AREA)
            dst = target_root / split / c / f'{src.stem}.png'
            cv2.imwrite(str(dst), img)
            counts[split][c] += 1

for c in classes:
    summary = ' | '.join(f'{split}={counts[split][c]}' for split in ['train', 'validation', 'test'])
    print(f'[{c}] {summary}')

print()
print(f'Dataset saved to: {target_root.resolve()}')
print(f'Total images: {sum(sum(v.values()) for v in counts.values())}')
if failures:
    print(f'Failed images: {len(failures)}')
    for p in failures[:10]:
        print('  ', p)


[0] train=144 | validation=31 | test=30
[1] train=144 | validation=31 | test=31
[2] train=144 | validation=31 | test=31
[3] train=144 | validation=31 | test=31
[4] train=145 | validation=31 | test=31
[5] train=145 | validation=31 | test=31
[6] train=145 | validation=31 | test=31
[7] train=144 | validation=31 | test=31
[8] train=146 | validation=31 | test=31
[9] train=143 | validation=31 | test=30

Dataset saved to: /Users/linxiao/Documents/ee6350/CNN_ACC/dataset_digits_gray64
Total images: 2062
